# Submit a Command Job

Generate a taxi-data summarization script, run it on the configured Azure ML compute, and inspect its terminal status and output URI.

**Sources:** Adapted from this repository's command-job patterns and the [Azure ML single-step examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/jobs/single-step), MIT License.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import dotenv_values
import requests
from azure.core.credentials import TokenCredential
from azure.ai.ml.entities import AmlCompute
import ast
import textwrap
from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedIdentityConfiguration

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
SUBSCRIPTION_ID = (env["AZURE_SUBSCRIPTION_ID"] or "").strip()
RESOURCE_GROUP = (env["AZURE_RESOURCE_GROUP"] or "").strip()
WORKSPACE_NAME = (env["AZUREML_WORKSPACE_NAME"] or "").strip()
REGISTRY_NAME = (env["AZUREML_REGISTRY_NAME"] or "").strip()

if len(env) != 4 or not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in only the four workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

credential = AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}

def arm_get(credential: TokenCredential, resource_id: str, api_version: str) -> dict:
    response = requests.get(
        f"https://management.azure.com{resource_id}",
        params={"api-version": api_version},
        headers={"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"},
        timeout=60,
    )
    response.raise_for_status()
    document = response.json()
    if not isinstance(document, dict):
        raise ValueError(f"Expected an ARM resource object for {resource_id}.")
    return document

def resolve_cluster(
    client: MLClient,
    credential: TokenCredential,
    compute_name: str | None = None,
) -> tuple[str, str]:
    if compute_name:
        compute = client.compute.get(compute_name)
    else:
        clusters = [item for item in client.compute.list() if isinstance(item, AmlCompute)]
        if len(clusters) != 1:
            raise ValueError(
                f"Expected one compute cluster, found {len(clusters)}. "
                "Set COMPUTE_NAME explicitly in this notebook's configuration cell."
            )
        name = clusters[0].name
        if not name:
            raise ValueError("The compute cluster has no name.")
        compute = client.compute.get(name)

    if not isinstance(compute, AmlCompute) or not compute.name:
        raise ValueError("COMPUTE_NAME must select a named AML compute cluster.")
    identities = compute.identity.user_assigned_identities if compute.identity else []
    if not identities or len(identities) != 1:
        raise ValueError(f"Cluster {compute.name} must have exactly one user-assigned managed identity.")
    identity = identities[0]
    client_id = identity.client_id
    if not client_id and identity.resource_id:
        resource = arm_get(credential, identity.resource_id, "2023-01-31")
        client_id = resource.get("properties", {}).get("clientId")
    if not isinstance(client_id, str) or not client_id:
        raise ValueError(f"Unable to resolve the client ID for cluster {compute.name}'s identity.")
    return compute.name, client_id

COMPUTE_NAME = None  # Set a name here only when the workspace has multiple clusters.
COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID = resolve_cluster(ml_client, credential, COMPUTE_NAME)
OUTPUT_DATASTORE = ml_client.datastores.get_default().name

EXPERIMENT_NAME = "workspace-readiness"

In [ ]:
generated_code_dir = SUITE_ROOT / "outputs/generated/foundations/command_job/code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
script_path = generated_code_dir / "summarize.py"
script_source = r'''
import argparse
import json
from pathlib import Path

import pandas as pd


def resolve_csv(path):
    path = Path(path)
    if path.is_file():
        return path
    matches = sorted(path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CSV beneath {path}, found {len(matches)}")
    return matches[0]


parser = argparse.ArgumentParser(description="Summarize a taxi CSV")
parser.add_argument("--input-data", required=True)
parser.add_argument("--output-data", required=True)
args = parser.parse_args()
source_path = resolve_csv(args.input_data)
frame = pd.read_csv(source_path)
output_dir = Path(args.output_data)
output_dir.mkdir(parents=True, exist_ok=True)
summary = {
    "source_file": source_path.name,
    "row_count": len(frame),
    "column_count": len(frame.columns),
    "columns": frame.columns.tolist(),
}
(output_dir / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
'''
script_source = textwrap.dedent(script_source).lstrip()
ast.parse(script_source, filename=str(script_path))
script_path.write_text(script_source, encoding="utf-8")
print(f"Generated command script: {script_path}")

job = command(
    display_name="Readiness taxi CSV summary",
    experiment_name=EXPERIMENT_NAME,
    code=str(generated_code_dir),
    command=(
        "python summarize.py "
        "--input-data ${{inputs.input_data}} "
        "--output-data ${{outputs.output_data}}"
    ),
    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=str(SAMPLE_ROOT / "data/taxi/raw/yellowTaxiData.csv"),
        )
    },
    outputs={
        "output_data": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount")
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute=COMPUTE_NAME,
    identity=ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID),
    tags={"validation": "workspace-readiness", "operation": "command-job"},
)

assert isinstance(job.identity, ManagedIdentityConfiguration)
assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")

submitted_job = ml_client.jobs.create_or_update(job)
print(f"Submitted job: {submitted_job.name}")
ml_client.jobs.stream(submitted_job.name)
final_job = ml_client.jobs.get(submitted_job.name)
if final_job.status != "Completed":
    raise RuntimeError(f"Job ended with status {final_job.status}")

output_datastore = OUTPUT_DATASTORE
output_uri = (
    f"azureml://datastores/{output_datastore}/paths/azureml/"
    f"{final_job.name}/output_data/"
)
download_dir = SUITE_ROOT / "outputs/foundation_command_job" / final_job.name
ml_client.jobs.download(
    final_job.name,
    output_name="output_data",
    download_path=download_dir,
)
summary_path = download_dir / "named-outputs/output_data/summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Expected downloaded output at {summary_path}")

print(f"Studio: {final_job.studio_url}")
print(f"Output URI: {output_uri}")
print(f"Downloaded summary: {summary_path}")


## Expected Result

The command job completes on the configured cluster and publishes a `summary.json` artifact in its output folder.

Next: `06_deploy_online_endpoint.ipynb`.